<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [8]:
from fastai import *
import pandas as pd
import os
import zipfile
import numpy as np
from dotenv import load_dotenv
import yfinance as yf
from sklearn.preprocessing import LabelEncoder
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


def build_dataset(time_period):


    dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
    path = 'ticker_list'
    kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)

    tk = pd.read_csv('ticker_list/sp500_companies.csv')

    tickers = tk['symbol'].tolist()
    tickers[0:25]
    len(tickers)
    tickers[499:503]
    df = yf.download(tickers, period=time_period)

    closes = df["Close"]
    returns = closes.pct_change() *100

    volumes = df['Volume']

    cs = closes.stack().rename("Close")
    rs = returns.stack().rename("Returns")
    vs = volumes.stack().rename("Volume")
    fdf = pd.concat([cs,rs,vs], axis = 1).reset_index()

    fdf.columns = ['date', 'ticker', 'close', 'return', 'volume']
    fdf = fdf.dropna(subset=['return'])

    fdf.sort_values(by=['ticker', 'date'], inplace=True)
    fdf.sort_values(['ticker', 'date'], inplace=True)
    df=fdf
    len(df)

    df['Lag 1'] = df.groupby('ticker')['close'].shift(1)
    df['Lag 2'] = df.groupby('ticker')['close'].shift(2)
    df['Lag 3'] = df.groupby('ticker')['close'].shift(3)
    df['Lag 5'] = df.groupby('ticker')['close'].shift(5)
    df['SMA 5'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=5).mean())
    df['SMA 10'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=10).mean())
    df['SMA 20'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=20).mean())
    df['SMA 30'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=30).mean())
    df['SMA 50'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=50).mean())

    df['volatility 5'] = df.groupby('ticker')['return'].transform(lambda s: s.rolling(5).std())
    df['volatility 10'] = df.groupby('ticker')['return'].transform(lambda s: s.rolling(10).std())
    df['volatility 20'] = df.groupby('ticker')['return'].transform(lambda s: s.rolling(20).std())
    df['volatility 30'] = df.groupby('ticker')['return'].transform(lambda s: s.rolling(30).std())

    df['Price change per volume'] = 1000000*(df['return']/df['volume'])

    df['id'] = LabelEncoder().fit_transform(df['ticker'])

    df['mk cap'] = df['close'] * df['volume'] / 100000000
    df = df.dropna()

    len(df)
    df['future return'] = df.groupby('ticker')['return'].shift(-1)
    df = df.dropna()


    return df

In [9]:
df = build_dataset("36mo")

Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 101M/101M [00:01<00:00, 80.4MB/s]


/tmp/ipykernel_5888/1728702025.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period=time_period)
[*********************100%***********************]  503 of 503 completed
/tmp/ipykernel_5888/1728702025.py:32: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100
/tmp/ipykernel_5888/1728702025.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['future return'] = df.groupby('ticker')['return'].shift(-1)


In [18]:
device = torch.device("cpu")

features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50','volume','Price change per volume','id','mk cap']
x = df[features]
target = 'future return'

In [14]:
def make_sequences(window, ticker_col='ticker', date_col='date'):
    X_list, y_list = [], []

    for ticker, group in df.groupby(target):
        group = group.sort_values(date_col)
        feat_arr = group[features].values
        target_arr = group[ticker_col].values

        n_days = len(group)
        for i in range(n_days - window):
            X_list.append(feat_arr[i : i + window])
            y_list.append(target_arr[i + window])
    return np.array(X_list), np.array(y_list)

In [15]:
X_seq, y_seq = make_sequences(10)
X_seq.shape
y_seq.shape

AttributeError: 'numpy.ndarray' object has no attribute 'shaped'

In [23]:
train_pct = 0.8
X_train, y_train = [], []
X_test, y_test = [], []
for ticker, group in df.groupby('ticker'):
    group = group.sort_values('date')
    feat_arr = group[features].values
    target_arr = group[target].values

    n_days = len(group)


    ticker_X, ticker_y = [], []
    for i in range(n_days - 10 ): #10 is the window, be sure to change it if you change the window
         ticker_X.append(feat_arr[i : i + 10])
         ticker_y.append(target_arr[i + 10])

    n_seqs = len(ticker_X)
    split_idx = int(train_pct * n_seqs)

    X_train.extend(ticker_X[:split_idx])
    y_train.extend(ticker_y[:split_idx])
    X_test.extend(ticker_X[split_idx:])
    y_test.extend(ticker_y[split_idx:])

x_train = np.array(X_train)
y_train = np.array(y_train)
x_test = np.array(X_test)
y_test = np.array(y_test)

print("train:", x_train.shape, y_train.shape)
print("test: ", x_test.shape, y_test.shape)

train: (275943, 10, 18) (275943,)
test:  (69361, 10, 18) (69361,)


In [24]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(275943, 10, 18)
(275943,)
(69361, 10, 18)
(69361,)


In [27]:
import numpy as np
for i, f in enumerate(features):
    n_inf = np.isinf(x_train[:, :, i]).sum()
    if n_inf > 0:
        print(f, n_inf)

In [28]:
n_features = x_train.shape[2]

scaler = StandardScaler()
scaler.fit(x_train.reshape(-1, n_features))

x_train_scaled = scaler.transform(x_train.reshape(-1, n_features)).reshape(x_train.shape)
x_test_scaled = scaler.transform(x_test.reshape(-1, n_features)).reshape(x_test.shape)

print(x_train_scaled.shape)
x_test_scaled.shape

(275943, 10, 18)


(69361, 10, 18)

In [29]:
from torch.utils.data import Dataset, DataLoader

class StockSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = StockSequenceDataset(x_train_scaled, y_train)
test_ds = StockSequenceDataset(x_test_scaled, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

xb, yb = next(iter(train_loader))
print(xb.shape)
yb.shape

torch.Size([256, 10, 18])


torch.Size([256, 1])

In [30]:
class StockLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.rnn = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.rnn(x)
        last_step = out[:, -1, :]
        return self.fc(last_step)

model = StockLSTM(input_size=18, hidden_size=64, num_layers=1).to(device)

In [31]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 20
train_losses, test_losses = [], []

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    train_loss = running_loss / len(train_ds)

    model.eval()
    running_test_loss = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            running_test_loss += loss.item() * xb.size(0)
    test_loss = running_test_loss / len(test_ds)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    print(f"Epoch {epoch+1}")
    print(train_loss)
    print(test_loss)

Epoch 1
4.506143448440018
6.580822423743902
Epoch 2
4.449152685218566
6.5826941654052815
Epoch 3
4.393906172480606
6.6123309054246615
Epoch 4
4.355447401915944
6.598151552617173
Epoch 5
4.329397931650542
6.651633312310973
Epoch 6
4.302837822579615
6.640376674611787
Epoch 7
4.282778319418833
6.654122236403652
Epoch 8
4.261412969416818
6.659070821124029
Epoch 9
4.243871686516848
6.694952555022158
Epoch 10
4.22089357319212
6.704319859432364
Epoch 11
4.199432768224354
6.75890692811261
Epoch 12
4.176502496863042
6.700165771530243
Epoch 13
4.15677851775207
6.790620094664831
Epoch 14
4.139546055659892
6.764976338149097
Epoch 15
4.116345089573317
6.793207905576128
Epoch 16
4.094209941464625
6.775744415209836
Epoch 17
4.07440330848001
6.843883444777176
Epoch 18
4.052123479518305
6.890405789349168
Epoch 19
4.031970363472798
6.926428553200832
Epoch 20
4.009413204275093
6.919333051609829
